In [52]:
import requests
import csv
import json
from datetime import datetime, timedelta
import time

import pandas as pd
import numpy as np

# Сбор данных

wall.get - Основной метод для получения записей со стены пользователя или сообщества вконтакте.

Для отправки API-запросов используются адреса следующего вида:
https://<адрес-сервера>/method/<имя-API-метода>?<параметры>
В этом выражении:
- <адрес-сервера> — адрес API ВКонтакте: api.vk.com.
- <имя-API-метода> — имя раздела и API-операции для вызова, например users.get или likes.add.
- <параметры> — параметры, которые передаются методу в строке запроса, например ...?v=5.199&p1=v1.

# Скрипт по количеству

In [ ]:
# функция для получения указанного количества постов со стены сообщества
def get_wall_posts():
    # указываем сервисный ключ, который необходим для выполнения запросов vk_api
    token = input()
    # указываем актуальную версию API
    version = 5.199
    # указываем domain - Короткий адрес пользователя или сообщества.
    domain = 'sber'
    # offset - Смещение, необходимое для выборки определённого подмножества записей.
    offset = 1
    # создаем переменную count - Максимальное число записей за один запрос
    count = 100
    # создаем пустой словарь для всех записей на стене
    all_posts = []
        
    while offset < 1000:
        response = requests.get('https://api.vk.com/method/wall.get', params = {
                            'access_token': token,
                            'v': version,
                            'domain': domain,
                            'count': count,
                            'offset': offset
                            })
        # преобразуем данные get запроса в словарь json
        data = response.json()['response']['items']
        offset += count
        # Пауза между запросами, чтобы не превысить лимиты API
        time.sleep(0.34)
        # расширяем список на очередные count постов
        all_posts.extend(data)
        
    # возвращаем список словарей
    return all_posts

# функция для записи данных в json файл
def get_json(all_posts):
    # открываем файл на запись
    with open('sber_wall.json', 'w') as file:
        # записываем данные в файл
        json.dump(all_posts, file, indent=3)

# вызываем функцию, которая создает json, и передаем в нее результат функции get_wall_posts
get_json(get_wall_posts())

# функция для записи нужных полей из json-ответа в csv файл
def file_writer(all_posts):
    with open('sber_wall.csv', 'w', encoding='UTF-8') as file:
        # создаем переменную, которая записывает данные в файл
        data_writer = csv.writer(file, delimiter=',')
        # записываем строку вида (это будут заголовки таблицы)
        data_writer.writerow(('post_id', 'post_date', 'likes_count', 'comments_count', 'views_count', 'reposts_count', 'type', 'post_text'))
        # разбираем json с помощью цикла
        for post in all_posts:
            try:
                # есть записи, где список attachments пустой, что приводит к ошибке, поэтому проверяем его с помощью условия 
                if len(post['attachments']) > 0:
                    type_post = post['attachments'][0]['type']
                else:
                    type_post = 'unknown'
            except:
                pass
            data_writer.writerow((post['id'], datetime.fromtimestamp(post['date']), post['likes']['count'], 
                                 post['comments']['count'], post['views']['count'], post['reposts']['count'],
                                 type_post, post['text'].replace('\n', ' ')))
    # возвращает путь к файлу
    return 'sber_wall.csv'

# вызываем функцию, которая создает csv файл, и передаем в нее результат функции get_wall_posts
file_writer(get_wall_posts())

# функция для записи csv файла в Dataframe pandas
def csv_to_df(csv_path):
    df = pd.read_csv(csv_path)
    return df

# вызываем функцию, которая читает csv файл в pandas dataframe, и передаем в нее результат функции file_writer
df = csv_to_df(file_writer(get_wall_posts()))

# Скрипт по дате

In [53]:
# функция для получения постов со стены сообщества за указанный период
def get_wall_posts(token, domain, start_date_str, end_date_str=None):
    # указываем актуальную версию API
    version = 5.199
    # offset - Смещение, необходимое для выборки определённого подмножества записей.
    offset = 0
    # создаем переменную count - Максимальное число записей за один запрос
    count = 100
    # создаем пустой словарь для всех записей на стене
    all_posts_data = []

    # Переводим строки в даты
    start_date = datetime.strptime(start_date_str, '%Y-%m-%d')
    end_date = (datetime.strptime(end_date_str, '%Y-%m-%d') + timedelta(days=1) - timedelta(seconds=1)) if end_date_str else datetime.datetime.now()

    going_flag = True
        
    while going_flag:
        response = requests.get('https://api.vk.com/method/wall.get', params = {
                            'access_token': token,
                            'v': version,
                            'domain': domain,
                            'count': count,
                            'offset': offset
                            })
        
        if 'error' in response.json():
            print("Ошибка API:", response.json()['error']['error_msg'])
            break

        # преобразуем данные get запроса в словарь json
        items = response.json()['response']['items']

        # Если посты закончились
        if not items:
            break

        # для каждого поста достаем дату
        for post in items:
            post_date = datetime.fromtimestamp(post['date'])

            # Поскольку ВК собирает посты от новых к старым, то останавливаемся, когда дойдем до даты старта
            if post_date < start_date:
                going_flag = False
                break

            if start_date <= post_date <= end_date:
               attachments = post.get('attachments', [])
               # есть записи, где список attachments пустой, что приводит к ошибке, поэтому проверяем его с помощью условия 
               post_type = attachments[0].get('type', 'unknown') if attachments else 'unknown' 

               # собираем данные в список словарей
               all_posts_data.append({
                    'post_id': post['id'],
                    'post_date': post_date,
                    'likes_count': post.get('likes', {}).get('count', 0),
                    'comments_count': post.get('comments', {}).get('count', 0),
                    'views_count': post.get('views', {}).get('count', 0),
                    'reposts_count': post.get('reposts', {}).get('count', 0),
                    'type': post_type,
                    'post_text': post.get('text', '').replace('\n', ' ')
                })

        # Берем следующие 100 постов на каждой итерации
        offset += count
        # Пауза между запросами, чтобы не превысить лимиты API
        time.sleep(0.34)

    # преобразуем список словарей в датафрейм
    df = pd.DataFrame(all_posts_data)
    df = df.sort_values(by='post_date', ascending=False).reset_index().drop('index', axis=1)
    print(f'Собрано {len(df)} постов.')
    # возвращаем датафрейм
    return df

In [54]:
# указываем сервисный ключ, который необходим для выполнения запросов vk_api
TOKEN = input("Введите токен API ВК: ").strip()
# указываем domain - Короткий адрес пользователя или сообщества.
DOMAIN = 'sber'

# указываем стартовую и конечную даты
START_DATE = '2025-07-25'
END_DATE = '2026-08-30'

# Собираем данные
df = get_wall_posts(
    token=TOKEN,
    domain=DOMAIN,
    start_date_str=START_DATE,
    end_date_str=END_DATE
)

# Сохраняем в CSV
if not df.empty:
    df.to_csv('sber_wall_2.csv', index=False, encoding='utf-8')

Собрано 1004 постов.
